In [1]:
import os, sys
import itertools
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics import root_mean_squared_error as rmse
parts=os.getcwd().split('/')
print(parts)
if (parts[-1]=='scripts') |(parts[-1]=='notebooks'):
    parts=parts[:-1]
srcpath='/'.join(parts)+'/results/'
dname_all=['diabetes', 'surgical', 'banking', 'adult', 'criteo','breastcancer']
dname=dname_all[5]
exp_type='random'
respath='%s/%s_%s_unlearning_swish.csv'%(srcpath, dname, exp_type)
#'%s/%s_%s_unlearning_0.csv'%(srcpath, dname, exp_type)
res_df=pd.read_csv(respath, sep='\t')
code_path='/'.join(parts)+'/scripts/'
sys.path.append(code_path)
from experiment_utils import dataset_health
modelpath='/'.join(parts)+'/models/'
from experiment_utils import data_norm_log
from unlearning.unlearn_eval import *
Xtrain, Ytrain, Xtest, Ytest, features=dataset_health(dname)
zXtrain, zXtest=data_norm_log(Xtrain, Xtest)
#zXtrain, zXtest=data_norm_log(Xtrain[features], Xtest[features])
res_df.rename(columns={'num_unlearned_samples':'n_unlearned'}, inplace=True)
colnames=list(res_df.columns)
print(len(colnames))
exp_model_specs=[ 'iter','solver_type',  'prot_per_class']
model_perf_cols=['et_retrain', 'et_unlearn', 'n_retrain', 'n_unlearned', 'dev_M0M2','dev_M1M2',
               'tr_M0_Bacc', 'tr_M1_Bacc', 'tr_M2_Bacc', #'tr_M0_AUC', 'tr_M1_AUC',  'tr_M2_AUC',
               'tr_M0_nAcc', 'tr_M1_nAcc','tr_retain_corr_M1', 'tr_imp_corr_M1',  'tr_lost_preds_M1',
               'tr_M2_nAcc', 'tr_imp_corr_M2',  'tr_retain_corr_M2','tr_lost_preds_M2' ,
               'te_M0_Bacc', 'te_M1_Bacc', 'te_M2_Bacc', #'te_M0_AUC', 'te_M1_AUC',  'te_M2_AUC',
               'te_M0_nAcc', 'te_M1_nAcc','te_retain_corr_M1', 'te_imp_corr_M1', 'te_lost_preds_M1',
               'te_M2_nAcc', 'te_imp_corr_M2',  'te_retain_corr_M2','te_lost_preds_M2',
               'un_M0_nAcc', 'un_M1_nAcc','un_retain_corr_M1', 'un_imp_corr_M1', 'un_lost_preds_M1',
               'un_M2_nAcc', 'un_imp_corr_M2',  'un_retain_corr_M2','un_lost_preds_M2']
model_param_cols=['et_retrain', 'et_unlearn', 'n_retrain', 'n_unlearned', 
               'prot_dev_01', 'prot_dev_02', 'prot_dev_12' ]
res_df.head(3)
res_df_time=res_df[exp_model_specs+['et_retrain', 'et_unlearn', 'n_retrain', 'n_unlearned']]

['', 'mnt', 'nvme1n1p2', 'codes_datasets', 'toolboxes', 'unlearn-lvq', 'notebooks']
/mnt/nvme1n1p2/codes_datasets/toolboxes/unlearn-lvq
55


In [2]:
#%autoreload 2
import sklvq
out_path='/'.join(parts)+'/results/metaanalysis/'
dist_func=sklvq.distances.SquaredEuclidean()
modelpath='/'.join(parts)+'/models/'
nprot_opts=[1,2]
cout=0
for nprots_per_class in nprot_opts:
    for solver_type in ['sgd']: #,'wgd','lbfgs']:
        cint=0
        picklefilename='%s%s/%s_random_swish_%s_nprot%d0.pkl'%(modelpath, dname, dname, solver_type, nprots_per_class)
        #picklefilename='%s%s/%s_random_%s_nprot%d0.pkl'%(modelpath, dname, dname, solver_type, nprots_per_class)
       # picklefilename='%s%s/%s_random_%s_nprot%d0.pkl'%(modelpath, dname, dname, solver_type, nprots_per_class)
        with open(picklefilename, 'rb') as file:
            model_sets=pickle.load(file)
            retrained_all, unlearned_all=model_sets['retrained'], model_sets['unlearned']
            M0=model_sets['original']
        print(picklefilename, M0.prototypes_.shape, zXtrain.shape)
        for nopts in list(unlearned_all.keys()):
            cint=0
            n=unlearned_all[nopts]['n']
            for iter in unlearned_all[nopts]['models'].keys():#)
                M1, M2=retrained_all[nopts]['models'][iter]['model'], unlearned_all[nopts]['models'][iter]['model']
                unlearn_indices=unlearned_all[nopts]['models'][iter]['unlearn_indices']
                retrain_indices=retrained_all[nopts]['models'][iter]['retrain_indices']
                zXretrain, zXunlearn=zXtrain.iloc[retrain_indices], zXtrain.iloc[unlearn_indices]
                Yretrain, Yunlearn=Ytrain[retrain_indices], Ytrain[unlearn_indices]
                est_train_M0, est_test_M0, est_unlearn_M0=M0.predict(zXretrain), M0.predict(zXtest), M0.predict(zXunlearn)
                est_train_M1, est_test_M1, est_unlearn_M1=M1.predict(zXretrain), M1.predict(zXtest), M1.predict(zXunlearn)
                est_train_M2, est_test_M2, est_unlearn_M2=M2.predict(zXretrain), M2.predict(zXtest), M2.predict(zXunlearn)
                data_dict_unlearn,data_dict_retrain={'zX_M1':zXunlearn}, {'zX_M1':zXretrain} #zXretrain
                data_dict_test={'zX_M1':zXtest}
                perf_unlearn=compare_perf_3(M0, M1, M2, data_dict_unlearn, Yunlearn)
                perf_retrain=compare_perf_3(M0, M1, M2, data_dict_retrain, Yretrain)
                perf_test=compare_perf_3(M0, M1, M2, data_dict_test, Ytest)
                lab_indices_1d=np.array([iter*np.ones(len(unlearn_indices)),n*np.ones(len(unlearn_indices)),unlearn_indices,Yunlearn, 
                                         est_unlearn_M0,est_unlearn_M1, est_unlearn_M2]).T
                dist_rel_M0, dist_rel_M1,dist_rel_M2=dist_func(zXunlearn, M0), dist_func(zXunlearn, M1), dist_func(zXunlearn, M2)
                unlearn_perf_stats=np.concatenate([lab_indices_1d, dist_rel_M0,dist_rel_M1, dist_rel_M2], axis=1)
                et_unlearn=res_df_time['et_unlearn'].loc[(res_df_time['solver_type']==solver_type) & (res_df_time['iter']==iter) &
                (res_df_time['n_unlearned']==n) & (res_df_time['prot_per_class']==nprots_per_class)].values[0]
                et_retrain=res_df_time['et_retrain'].loc[(res_df_time['solver_type']==solver_type) & (res_df_time['iter']==iter) &
                (res_df_time['n_unlearned']==n) & (res_df_time['prot_per_class']==nprots_per_class)].values[0]
                dev01, max_dev_indx0=compare_fidelity_glvq(M0, M1)
                dev02, max_dev_indx0=compare_fidelity_glvq(M0, M2)
                dev12, max_dev_indx0=compare_fidelity_glvq(M1, M2)
                compare_dict_un={'prot_per_class': nprots_per_class,'n_unlearned': len(unlearn_indices), 'iter':iter,
                    'solver_type':solver_type, 'et_retrain':et_retrain, 'et_unlearn':et_unlearn,
                    'dev_M0M1':dev01, 'dev_M0M2':dev02, 'dev_M1M2':dev12,
                    'n_retrain': len(retrain_indices),'un_M0_nAcc': perf_unlearn['M0_npreds'], 'un_M1_nAcc': perf_unlearn['M1_npreds'], 
                    'un_retain_corr_M1':perf_unlearn['ret_corr_pred_M1'], 'un_lost_preds_M1':perf_unlearn['lost_corr_pred_M1'],
                    'un_imp_corr_M1':perf_unlearn['imp_corr_pred_M1'], 'un_M2_nAcc': perf_unlearn['M2_npreds'],  
                    'un_retain_corr_M2':perf_unlearn['ret_corr_pred_M2'], 'un_lost_preds_M2':perf_unlearn['lost_corr_pred_M2'],
                    'un_imp_corr_M2':perf_unlearn['imp_corr_pred_M2'], 
                    'un_M0_Bacc': perf_unlearn['M0_Bacc'],'un_M1_Bacc': perf_unlearn['M1_Bacc'],'un_M2_Bacc': perf_unlearn['M2_Bacc'],
                    'tr_M0_nAcc': perf_retrain['M0_npreds'], 'tr_M1_nAcc': perf_retrain['M1_npreds'], 
                    'tr_retain_corr_M1':perf_retrain['ret_corr_pred_M1'], 'tr_lost_preds_M1':perf_retrain['lost_corr_pred_M1'],
                    'tr_imp_corr_M1':perf_retrain['imp_corr_pred_M1'], 'tr_M2_nAcc': perf_retrain['M2_npreds'],  
                    'tr_retain_corr_M2':perf_retrain['ret_corr_pred_M2'], 'tr_lost_preds_M2':perf_retrain['lost_corr_pred_M2'],
                    'tr_imp_corr_M2':perf_retrain['imp_corr_pred_M2'], 
                    'tr_M0_Bacc': perf_retrain['M0_Bacc'],'tr_M1_Bacc': perf_retrain['M1_Bacc'],'tr_M2_Bacc': perf_retrain['M2_Bacc'],
                    'n_test':len(Ytest), 'te_M0_nAcc': perf_test['M0_npreds'], 'te_M1_nAcc': perf_test['M1_npreds'], 
                    'te_retain_corr_M1':perf_test['ret_corr_pred_M1'], 'te_lost_preds_M1':perf_test['lost_corr_pred_M1'],
                    'te_imp_corr_M1':perf_test['imp_corr_pred_M1'], 'te_M2_nAcc': perf_test['M2_npreds'],  
                    'te_retain_corr_M2':perf_test['ret_corr_pred_M2'], 'te_lost_preds_M2':perf_test['lost_corr_pred_M2'],
                    'te_imp_corr_M2':perf_test['imp_corr_pred_M2'], 
                    'te_M0_Bacc': perf_test['M0_Bacc'],'te_M1_Bacc': perf_test['M1_Bacc'],'te_M2_Bacc': perf_test['M2_Bacc']}
                if cout==0:
                    compare_df=pd.DataFrame.from_dict(data=compare_dict_un, orient='index').T
                    cout+=1
                else:
                    temp_0=pd.DataFrame.from_dict(data=compare_dict_un, orient='index').T
                    compare_df=pd.concat([compare_df, temp_0])
                    cout+=1
                if (cint==0):
                    dist2prots_df=pd.DataFrame(data=unlearn_perf_stats, columns=['iter', 'n_unlearned', 'unlearn_indices',
                            'Yunlearn_gt','Yunlearn_M0', 'Yunlearn_M1', 'Yunlearn_M2']+\
                                        ['M0_dist_prot_'+ str(i) for i in list(range(0,dist_rel_M0.shape[1]))]+\
                                        ['M1_dist_prot_'+ str(i) for i in list(range(0,dist_rel_M0.shape[1]))]+\
                                        ['M1_dist_prot_'+ str(i) for i in list(range(0,dist_rel_M0.shape[1]))])
                    dist2prots_df.insert(0, 'solver_type', solver_type)
                    cint+=1
                else:
                    temp_df=pd.DataFrame(data=unlearn_perf_stats, 
                                columns=['iter', 'n_unlearned', 'unlearn_indices', 'Yunlearn_gt',
                                         'Yunlearn_M0', 'Yunlearn_M1', 'Yunlearn_M2']+\
                                        ['M0_dist_prot_'+ str(i) for i in list(range(0,dist_rel_M0.shape[1]))]+\
                                        ['M1_dist_prot_'+ str(i) for i in list(range(0,dist_rel_M0.shape[1]))]+\
                                        ['M1_dist_prot_'+ str(i) for i in list(range(0,dist_rel_M0.shape[1]))])
                    temp_df.insert(0, 'solver_type', solver_type)
                    dist2prots_df=pd.concat([dist2prots_df, temp_df])
                    cint+=1                  
                del est_train_M0, est_train_M1, est_train_M2, est_test_M0, est_test_M1, est_test_M2
                del est_unlearn_M0, est_unlearn_M1, est_unlearn_M2, Yretrain, Yunlearn
        dist2prots_df['unlearn_indices']=dist2prots_df['unlearn_indices'].astype(int)    
        metapath='%s%s/%s_%s_dist_presd_nprot_swish_%d.csv'%(out_path,solver_type, dname, exp_type, nprots_per_class)
        dist2prots_df.to_csv(metapath, sep='\t', index=False)
#compare_df.to_csv('%smeta_%s_random_unlearned.csv'%(out_path, dname), index=False, sep='\t')
#res_df=pd.read_csv(respath, sep='\t')

/mnt/nvme1n1p2/codes_datasets/toolboxes/unlearn-lvq/models/breastcancer/breastcancer_random_swish_sgd_nprot10.pkl (2, 30) (381, 30)
/mnt/nvme1n1p2/codes_datasets/toolboxes/unlearn-lvq/models/breastcancer/breastcancer_random_swish_sgd_nprot20.pkl (4, 30) (381, 30)


In [3]:
compare_df[compare_df.columns[4:]]=compare_df[compare_df.columns[4:]].astype('float64')
solver_type='sgd'
sliced_df=compare_df[exp_model_specs+['n_test']+model_perf_cols].loc[(compare_df['solver_type']==solver_type)]
models, subsets=['M1', 'M2'], ['tr', 'te', 'un']
eval_metrics, nan_counter=[],[]
for subset in subsets:
    if subset==subsets[0]:
        sliced_df['speedupX']=sliced_df['et_retrain'].to_numpy()/sliced_df['et_unlearn'].to_numpy()
        eval_metrics.append('speedupX')
        sliced_df['%s_Acc_M0'%(subset)]=sliced_df['%s_M0_nAcc'%(subset)]/sliced_df['n_retrain']  #'n_test'
        eval_metrics.append('%s_Acc_M0'%(subset))
        eval_metrics.append('dev_M0M2')
        eval_metrics.append('dev_M1M2')
    elif subset=='te':
        sliced_df['%s_Acc_M0'%(subset)]=sliced_df['%s_M0_nAcc'%(subset)]/sliced_df['n_test']  #'n_test'
        eval_metrics.append('%s_Acc_M0'%(subset))
    else:
        sliced_df['%s_Acc_M0'%(subset)]=sliced_df['%s_M0_nAcc'%(subset)]/sliced_df['n_unlearned']  #'n_test'
        eval_metrics.append('%s_Acc_M0'%(subset))
    for model in models:
        if subset=='tr':
            sliced_df['%s_Acc_%s'%(subset, model)]=sliced_df['%s_%s_nAcc'%(subset, model)]/sliced_df['n_retrain']
        elif subset=='te':
            sliced_df['%s_Acc_%s'%(subset, model)]=sliced_df['%s_%s_nAcc'%(subset, model)]/sliced_df['n_test']
        else:
            sliced_df['%s_Acc_%s'%(subset, model)]=sliced_df['%s_%s_nAcc'%(subset, model)]/sliced_df['n_unlearned']
        eval_metrics.append('%s_Acc_%s'%(subset, model))
        if np.sum(sliced_df['%s_M0_nAcc'%subset]==0)==0:
            sliced_df['%s_retain_frac_%s'%(subset, model)]=sliced_df['%s_retain_corr_%s'%(subset, model)]/sliced_df['%s_M0_nAcc'%subset]*100
            sliced_df['%s_loss_frac_%s'%(subset, model)]=sliced_df['%s_lost_preds_%s'%(subset, model)]/sliced_df['%s_M0_nAcc'%subset]*100
            sliced_df['%s_gain_frac_%s'%(subset, model)]=sliced_df['%s_imp_corr_%s'%(subset, model)]/sliced_df['%s_M0_nAcc'%subset]*100
        else:   
            sliced_df['%s_retain_frac_%s'%(subset, model)]=sliced_df['%s_retain_corr_%s'%(subset, model)]/sliced_df['%s_M0_nAcc'%subset]*100
            sliced_df['%s_loss_frac_%s'%(subset, model)]=sliced_df['%s_lost_preds_%s'%(subset, model)]/sliced_df['%s_M0_nAcc'%subset]*100
            sliced_df['%s_gain_frac_%s'%(subset, model)]=sliced_df['%s_imp_corr_%s'%(subset, model)]/sliced_df['%s_M0_nAcc'%subset]*100
        sliced_df['nan_%s_retain_frac_%s'%(subset, model)]=sliced_df['%s_retain_frac_%s'%(subset, model)].isnull()
        sliced_df['nan_%s_loss_frac_%s'%(subset, model)]=sliced_df['%s_loss_frac_%s'%(subset, model)].isnull()
        sliced_df['nan_%s_gain_frac_%s'%(subset, model)]=sliced_df['%s_gain_frac_%s'%(subset, model)].isnull()
        eval_metrics.append('%s_retain_frac_%s'%(subset, model))
        eval_metrics.append('%s_loss_frac_%s'%(subset, model))
        eval_metrics.append('%s_gain_frac_%s'%(subset, model))
        nan_counter.append('%s_retain_frac_%s'%(subset, model))
        nan_counter.append('%s_loss_frac_%s'%(subset, model))
        nan_counter.append('%s_gain_frac_%s'%(subset, model))
eval_metrics=np.unique(eval_metrics)
nan_cols=np.array(['nan_'+ entry for entry in nan_counter])#nan_man_std=np.intersect1d
nan_res=sliced_df.groupby(['n_unlearned','solver_type','prot_per_class'])[nan_cols].sum().reset_index()
mean_res=sliced_df.groupby(['n_unlearned','solver_type','prot_per_class'])[eval_metrics].mean().reset_index()
std_res=sliced_df.groupby(['n_unlearned','solver_type','prot_per_class'])[eval_metrics].std().reset_index()
col_order=np.array(['n_unlearned', 'solver_type', 'prot_per_class', 'speedupX','dev_M0M2','dev_M1M2', 
           'tr_Acc_M0', 'tr_Acc_M1', 'tr_Acc_M2', 'tr_retain_frac_M1', 'tr_retain_frac_M2', 
           'tr_loss_frac_M1', 'tr_loss_frac_M2','tr_gain_frac_M1', 'tr_gain_frac_M2', 
           'te_Acc_M0','te_Acc_M1','te_Acc_M2', 'te_retain_frac_M1', 
           'te_retain_frac_M2','te_loss_frac_M1', 'te_loss_frac_M2', 'te_gain_frac_M1', 'te_gain_frac_M2',
           'un_Acc_M0','un_Acc_M1', 'un_Acc_M2', 'un_retain_frac_M1', 'un_retain_frac_M2',
           'un_loss_frac_M1', 'un_loss_frac_M2', 'un_gain_frac_M1', 'un_gain_frac_M2'])
mean_res=mean_res[col_order].copy()
mean_cols=['mean_'+ entry for entry in eval_metrics]
std_cols=['std_'+ entry for entry in eval_metrics]
mean_res.rename(columns=dict(zip(eval_metrics, mean_cols)), inplace=True)
std_res=std_res[col_order].copy()
std_res.rename(columns=dict(zip(eval_metrics, std_cols)), inplace=True)
common_cols=list(np.intersect1d(list(mean_res.columns), list(std_res.columns)))    
print(common_cols)
final_col_order=list(itertools.chain(*[ [mean_cols[i], std_cols[i]] for i in range(0, len(mean_cols))]))
combined=pd.concat([mean_res, std_res[std_cols]], axis=1)
combined=combined[common_cols+final_col_order]
for nan_col in nan_counter:
    comb_cols=['_'.join(x.split('_')[1:]) for x in list(combined.columns)]
    idx=np.where(nan_col==np.array(comb_cols))[0]
   # print(nan_col, idx, len(comb_cols))
    if cout==0:
        insert_loc=idx[0]+2
    else:
        insert_loc=idx[0]+2
    combined.insert(insert_loc,'nan_%s'%nan_col , nan_res['nan_%s'%nan_col])
metapath='%s%s/%s_%s_swish_summary.csv'%(out_path,solver_type, dname, exp_type)
print(metapath)
combined.to_csv(metapath, sep='\t', index=False)

['n_unlearned', 'prot_per_class', 'solver_type']
/mnt/nvme1n1p2/codes_datasets/toolboxes/unlearn-lvq/results/metaanalysis/sgd/breastcancer_random_swish_summary.csv


In [4]:

cols=['n_unlearned', 'prot_per_class', 'iter', 'n_retrain', 'n_test', 'te_M0_nAcc', 'te_M1_nAcc', 'te_M2_nAcc', 'un_M1_nAcc', 'un_M2_nAcc',
      'tr_retain_corr_M1', 'tr_retain_corr_M2', 'te_retain_corr_M1', 'te_retain_corr_M2','tr_M1_nAcc','tr_M2_nAcc']
compare_df[cols].loc[(compare_df['solver_type']=='sgd') & (compare_df['prot_per_class']==1)]#[combined.columns[1:]]

,n_unlearned,prot_per_class,iter,n_retrain,n_test,te_M0_nAcc,te_M1_nAcc,te_M2_nAcc,un_M1_nAcc,un_M2_nAcc,tr_retain_corr_M1,tr_retain_corr_M2,te_retain_corr_M1,te_retain_corr_M2,tr_M1_nAcc,tr_M2_nAcc
0,1,1,0,380.0,188.0,174.0,174.0,152.0,0.0,0.0,360.0,308.0,174.0,148.0,360.0,315.0
0,1,1,1,380.0,188.0,174.0,174.0,152.0,1.0,1.0,359.0,307.0,174.0,148.0,359.0,314.0
0,1,1,2,380.0,188.0,174.0,174.0,152.0,1.0,1.0,359.0,307.0,174.0,148.0,359.0,314.0
0,1,1,3,380.0,188.0,174.0,174.0,152.0,1.0,1.0,359.0,307.0,174.0,148.0,359.0,314.0
0,1,1,4,380.0,188.0,174.0,174.0,152.0,1.0,1.0,359.0,307.0,174.0,148.0,359.0,314.0
0,5,1,0,377.0,188.0,174.0,174.0,152.0,5.0,5.0,356.0,304.0,174.0,148.0,356.0,311.0
0,5,1,1,376.0,188.0,174.0,174.0,152.0,5.0,3.0,355.0,305.0,174.0,148.0,355.0,312.0
0,5,1,2,376.0,188.0,174.0,174.0,152.0,5.0,5.0,355.0,303.0,174.0,148.0,355.0,310.0
0,5,1,3,376.0,188.0,174.0,174.0,152.0,5.0,4.0,355.0,304.0,174.0,148.0,355.0,311.0
0,5,1,4,376.0,188.0,174.0,174.0,152.0,5.0,4.0,355.0,304.0,174.0,148.0,355.0,311.0


In [5]:
combined.loc[combined['n_unlearned']==1]
del combined

In [14]:
sliced_df[nan_cols]
sliced_df.groupby(['n_unlearned','solver_type','prot_per_class'])[nan_cols].sum().reset_index()

,n_unlearned,solver_type,prot_per_class,nan_tr_retain_frac_M1,nan_tr_loss_frac_M1,nan_tr_gain_frac_M1,nan_tr_retain_frac_M2,nan_tr_loss_frac_M2,nan_tr_gain_frac_M2,nan_te_retain_frac_M1,...,nan_te_gain_frac_M1,nan_te_retain_frac_M2,nan_te_loss_frac_M2,nan_te_gain_frac_M2,nan_un_retain_frac_M1,nan_un_loss_frac_M1,nan_un_gain_frac_M1,nan_un_retain_frac_M2,nan_un_loss_frac_M2,nan_un_gain_frac_M2
0,4,sgd,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,4,sgd,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,4,sgd,3,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,33,sgd,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,33,sgd,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,33,sgd,3,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,326,sgd,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,326,sgd,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,326,sgd,3,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,1629,sgd,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
